# Clase 143 — Transformers: arquitectura, BERT, GPT

El **Transformer** (Vaswani et al. 2017) reemplazó a las RNN. Su bloque es
`LN → MultiHeadAttention → +residual → LN → FFN → +residual`, más un **positional
encoding** porque la atención es permutation-invariant. Variantes: **BERT**
(encoder-only), **GPT** (decoder-only) y **T5** (encoder-decoder).

Requiere: `tensorflow` / `keras` (≥ 3.0), `numpy`. Se ejecuta en Colab con GPU.

## 1. Positional encoding sinusoidal (numpy)

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)
def positional_encoding(seq_len, d_model):
    pos = np.arange(seq_len)[:, None]
    i = np.arange(d_model)[None, :]
    ang = pos / np.power(10000.0, (2 * (i // 2)) / d_model)
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(ang[:, 0::2])   # dimensiones pares -> seno
    pe[:, 1::2] = np.cos(ang[:, 1::2])   # dimensiones impares -> coseno
    return pe

pe = positional_encoding(50, 64)
print("positional encoding:", pe.shape, "| rango:", round(pe.min(), 2), round(pe.max(), 2))

## 2. `MultiHeadAttention` como pieza central

In [ ]:
d_model, n_heads = 64, 4
x = keras.Input(shape=(20, d_model))
mha = layers.MultiHeadAttention(num_heads=n_heads, key_dim=d_model // n_heads)
print("salida MHA (self):", mha(x, x).shape)

## 3. Bloque encoder (Pre-LN + residuales)

In [ ]:
def bloque_encoder(d_model, n_heads, d_ff):
    entrada = keras.Input(shape=(None, d_model))
    a = layers.LayerNormalization()(entrada)                       # Pre-LN (estándar moderno)
    a = layers.MultiHeadAttention(n_heads, d_model // n_heads)(a, a)
    x = layers.Add()([entrada, a])                                 # conexión residual
    f = layers.LayerNormalization()(x)
    f = layers.Dense(d_ff, activation="relu")(f)                   # FFN, d_ff = 4 * d_model
    f = layers.Dense(d_model)(f)
    x = layers.Add()([x, f])                                       # residual
    return keras.Model(entrada, x, name="encoder_block")

enc = bloque_encoder(64, 4, 256)
enc.summary()

## 4. Bloque decoder con máscara causal (mini-GPT)

In [ ]:
def bloque_decoder(d_model, n_heads, d_ff):
    entrada = keras.Input(shape=(None, d_model))
    a = layers.LayerNormalization()(entrada)
    a = layers.MultiHeadAttention(n_heads, d_model // n_heads)(
        a, a, use_causal_mask=True)                    # GPT: cada token mira <= t
    x = layers.Add()([entrada, a])
    f = layers.LayerNormalization()(x)
    f = layers.Dense(d_ff, activation="gelu")(f)
    f = layers.Dense(d_model)(f)
    return keras.Model(entrada, layers.Add()([x, f]), name="decoder_block")

dec = bloque_decoder(64, 4, 256)
print("params del bloque decoder:", dec.count_params())

## 5. Mini-GPT: token embedding + positional embedding

In [ ]:
vocab, seq, d_model = 200, 32, 64
tokens = keras.Input(shape=(seq,))
tok_emb = layers.Embedding(vocab, d_model)(tokens)
posiciones = keras.ops.arange(seq)                       # 0..seq-1
pos_emb = layers.Embedding(seq, d_model)(posiciones)     # positional embedding aprendible
h = tok_emb + pos_emb                                    # broadcast (batch, seq, d)
h = bloque_decoder(d_model, 4, 256)(h)
logits = layers.Dense(vocab)(h)
mini_gpt = keras.Model(tokens, logits, name="mini_gpt")
print("mini-GPT salida:", mini_gpt.output_shape)         # (None, 32, 200)

## 6. Las 3 variantes: BERT / GPT / T5

In [ ]:
variantes = {
    "BERT (encoder-only)":  "MLM bidireccional -> clasificación, NER, similaridad",
    "GPT (decoder-only)":   "next-token + causal mask -> generación, LLMs modernos",
    "T5 (encoder-decoder)": "span-corruption -> traducción, resumen",
}
for k, v in variantes.items():
    print(f"- {k}: {v}")
# En producción: from transformers import AutoModel
#                AutoModel.from_pretrained("bert-base-uncased")

## Ejercicios

1. **Transformer block desde cero**: implementá `x = x + mha(LN(x)); x = x + ffn(LN(x))`.
2. **Positional encoding**: implementá la fórmula sinusoidal y visualizala como heatmap.
3. **Mini-GPT**: apilá 4 bloques decoder con causal mask y entrená next-token sobre texto.
4. **HuggingFace**: cargá `bert-base-uncased` y `gpt2` e inspeccioná `num_attention_heads`.

## Conclusiones

- El bloque Transformer es `LN → MHA → residual → LN → FFN → residual`; **Pre-LN** entrena mejor.
- El **positional encoding** (sinusoidal o aprendible) inyecta orden a una atención permutation-invariant.
- **BERT** es encoder-only (bidireccional, MLM); **GPT** decoder-only (causal, next-token); **T5** ambos.
- La FFN expande a `d_ff = 4·d_model`; el `Dense(vocab)` final produce los logits.
- Mejoras modernas (Flash Attention, RoPE, GQA, RMSNorm) se cubren en la clase 144.